# 00 · Data collection: forecasts and observations → daily snapshot

**Purpose.** Download, for every spot in the corpus, the Open-Meteo marine forecast (waves, swell, wind waves, sea level incl. tides), the Open-Meteo wind forecast, and the latest CWA buoy and tide-station observations. Save everything as one dated snapshot.

**Inputs**
- `data/spots/spots.csv`: spot IDs and coordinates (`lat`, `lon`, optional offshore `forecast_lat`, `forecast_lon`)
- `data/resources/cwa_stations_en.csv`: our lookup table that maps CWA station IDs to English names
- `.env`: `CWA_API_KEY` (never hard-coded)

**Outputs** (`data/snapshots/YYYY-MM-DD/`)
- `marine.parquet`: hourly marine forecast per spot
- `wind.parquet`: hourly wind forecast per spot
- `cwa_obs.parquet`: buoy and tide-station observations with English station names
- `raw/cwa_*.json.gz`: the unmodified CWA responses (so we can re-parse them later)
- `manifest.json`: fetch time, what succeeded/failed, attribution, parameters
- `collection.log`: the log of this run

**Why daily snapshots?** Forecasts change every few hours, so a query such as *"good beginner spot tomorrow"* has a different correct answer every day. Relevance judgments and all experiments therefore run against a *fixed* snapshot. Only then can results be reproduced, and baseline and improved systems be compared on identical data.

The same code runs unattended through `collect_snapshot.py` (see README → *Daily snapshots*). That script executes the code cells of this notebook, so there is a single source of truth.

In [ ]:
# ---- Configuration (the only cell you should need to edit) ----
import os
from pathlib import Path


def find_project_root(start=None):
    """Walk up from the working folder until we find the project root (contains notebooks/ and requirements.txt)."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / "notebooks").is_dir() and (folder / "requirements.txt").is_file():
            return folder
    raise FileNotFoundError("Project root not found. Open the project folder in VS Code (File > Open Folder).")


ROOT = find_project_root()
DATA_DIR = ROOT / "data"
SPOTS_FILE = DATA_DIR / "spots" / "spots.csv"
RESOURCES_DIR = DATA_DIR / "resources"
STATION_LOOKUP_FILE = RESOURCES_DIR / "cwa_stations_en.csv"
UNMAPPED_STATIONS_FILE = RESOURCES_DIR / "cwa_stations_unmapped.csv"
SNAP_ROOT = DATA_DIR / "snapshots"
ENV_FILE = ROOT / ".env"

# When another notebook loads this one with %run (e.g. 06_demo for live data), only the functions are defined.
LIBRARY_MODE = os.environ.get("SURF_IR_LIBRARY_MODE", "0") == "1"

TIMEZONE = "Asia/Taipei"      # all timestamps are stored as naive local Taiwan time
SNAPSHOT_DATE = None          # None = today's date in Taiwan; a "YYYY-MM-DD" string only re-labels the folder
SAMPLE_N = None               # e.g. 3 = fetch only the first 3 spots (quick development runs)
SEED = 42                     # not used for randomness here; kept for a uniform config across notebooks
OVERWRITE = False             # False = skip if today's snapshot already exists
FETCH_CWA = True              # False = skip CWA observations (e.g. while waiting for an API key)

FORECAST_DAYS = 7
REQUEST_TIMEOUT_S = 20
MAX_RETRIES = 4               # retries with exponential back-off for 429/5xx and connection errors

MARINE_URL = "https://marine-api.open-meteo.com/v1/marine"
WEATHER_URL = "https://api.open-meteo.com/v1/forecast"
MARINE_VARS = [
    "wave_height", "wave_direction", "wave_period",
    "swell_wave_height", "swell_wave_direction", "swell_wave_period",
    "wind_wave_height", "wind_wave_direction", "wind_wave_period",
    "sea_level_height_msl",
]
WIND_VARS = ["wind_speed_10m", "wind_direction_10m", "wind_gusts_10m", "temperature_2m"]   # temperature: demo forecast table

CWA_URL = "https://opendata.cwa.gov.tw/api/v1/rest/datastore/{dataset_id}"
CWA_DATASETS = {
    "obs_48h": "O-B0075-001",   # 48-hour buoy and tide-station observations (verified 2026-10-16)
}
# Official station list with English names and coordinates (file API, not the datastore API)
CWA_STATIONS_URL = "https://opendata.cwa.gov.tw/fileapi/v1/opendataapi/O-B0076-001"
REFRESH_STATION_LOOKUP = True  # update cwa_stations_en.csv from CWA's official station list on every run

## 1. Imports, logging and the API key
The key is read from `.env` with `python-dotenv` so it never ends up in Git or in a notebook output. Every failure is logged to a file inside the snapshot folder; a scheduled run has no one watching the screen.

In [ ]:
import gzip
import json
import ssl
import logging
import shutil
import time
from datetime import datetime
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import certifi
import requests
from dotenv import load_dotenv
from IPython.display import display
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

load_dotenv(ENV_FILE)
CWA_API_KEY = os.getenv("CWA_API_KEY", "").strip()

log = logging.getLogger("surf_ir.collect")
log.setLevel(logging.INFO)
if not log.handlers:
    _console = logging.StreamHandler()
    _console.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s", "%H:%M:%S"))
    log.addHandler(_console)


def attach_log_file(path):
    """Also write the log to a file (one file per snapshot)."""
    for h in list(log.handlers):
        if isinstance(h, logging.FileHandler):
            log.removeHandler(h)
            h.close()
    path.parent.mkdir(parents=True, exist_ok=True)
    fh = logging.FileHandler(path, encoding="utf-8")
    fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
    log.addHandler(fh)


def now_taiwan():
    return datetime.now(ZoneInfo(TIMEZONE)).replace(tzinfo=None, microsecond=0)


if not CWA_API_KEY:
    print("Note: CWA_API_KEY is not set in .env; CWA observations will be skipped.")

## 2. A robust HTTP session
Free APIs occasionally return 429 (rate limit) or 5xx errors. `urllib3.Retry` retries these with exponential back-off (1.5 s, 3 s, 6 s, …) before we give up and log the failure. One failed spot must not abort the whole snapshot. We also mask the API key in every message we log.

**CWA certificate and Python 3.13.** From Python 3.13, certificates are checked in a *strict* mode by default (`VERIFY_X509_STRICT`). The certificate chain of `opendata.cwa.gov.tw` lacks a field that this strict mode requires ("Missing Subject Key Identifier"), so every CWA request fails on Python 3.13 although it works on 3.11. For that one host only, we keep full certificate verification (trusted authority via `certifi`, hostname, expiry) and switch off just the extra strict check. All other hosts use the normal settings.

In [ ]:
class CWASSLAdapter(HTTPAdapter):
    """Normal certificate verification, minus Python 3.13's extra strict X.509 check (see above)."""

    def init_poolmanager(self, *args, **kwargs):
        ctx = ssl.create_default_context(cafile=certifi.where())
        ctx.verify_flags &= ~getattr(ssl, "VERIFY_X509_STRICT", 0)
        kwargs["ssl_context"] = ctx
        return super().init_poolmanager(*args, **kwargs)


def make_session():
    retry = Retry(
        total=MAX_RETRIES, backoff_factor=1.5,
        status_forcelist=[429, 500, 502, 503, 504], allowed_methods=["GET"],
        raise_on_status=False,
    )
    session = requests.Session()
    session.mount("https://", HTTPAdapter(max_retries=retry))
    session.mount("https://opendata.cwa.gov.tw", CWASSLAdapter(max_retries=retry))
    session.headers["User-Agent"] = "surf-ir-taiwan (MIS590 student project, non-commercial)"
    return session


def _mask(text):
    return str(text).replace(CWA_API_KEY, "***") if CWA_API_KEY else str(text)


def get_json(session, url, params, label):
    """GET a JSON document. Returns (data, error_message); exactly one of them is None."""
    try:
        resp = session.get(url, params=params, timeout=REQUEST_TIMEOUT_S)
        if resp.status_code != 200:
            msg = f"{label}: HTTP {resp.status_code}: {_mask(resp.text[:200])}"
            log.warning(msg)
            return None, msg
        return resp.json(), None
    except (requests.RequestException, ValueError) as exc:
        msg = f"{label}: {type(exc).__name__}: {_mask(exc)}"
        log.warning(msg)
        return None, msg

## 3. Spot locations
Only IDs and coordinates are needed here; the full profile is validated in `01_spot_corpus`. The marine model grid is coarse near Taiwan, and a beach coordinate can fall into a *land* grid cell (all values empty). Each spot can therefore have a separate offshore `forecast_lat`/`forecast_lon` for the marine request. Wind is requested at the beach itself.

In [ ]:
def load_spot_locations(path=SPOTS_FILE, sample_n=SAMPLE_N):
    spots = pd.read_csv(path, dtype={"spot_id": str})
    required = ["spot_id", "name", "lat", "lon"]
    missing = [c for c in required if c not in spots.columns]
    if missing:
        raise ValueError(f"{path} is missing columns: {missing}")
    for col in ["forecast_lat", "forecast_lon"]:
        if col not in spots.columns:
            spots[col] = np.nan
    spots["forecast_lat"] = spots["forecast_lat"].fillna(spots["lat"])
    spots["forecast_lon"] = spots["forecast_lon"].fillna(spots["lon"])
    spots = spots.dropna(subset=["lat", "lon"])
    if sample_n:
        spots = spots.head(sample_n)
    return spots[["spot_id", "name", "lat", "lon", "forecast_lat", "forecast_lon"]].reset_index(drop=True)

## 4. Open-Meteo marine and wind forecasts
One request per spot keeps failures isolated and the log readable (about 2 × 50 requests per day, far below Open-Meteo's free limit). `timezone=Asia/Taipei` makes the API return local times, so "tomorrow morning" in a query and the forecast hours use the same clock.

In [ ]:
def _hourly_frame(data, spot_id, variables):
    hourly = data.get("hourly", {})
    df = pd.DataFrame({"time": pd.to_datetime(hourly.get("time", []))})
    for var in variables:
        df[var] = pd.to_numeric(pd.Series(hourly.get(var, [None] * len(df))), errors="coerce")
    df.insert(0, "spot_id", spot_id)
    df["grid_lat"], df["grid_lon"] = data.get("latitude"), data.get("longitude")
    return df


def fetch_marine(session, spots, on_step=None):
    frames, failed = [], {}
    for s in spots.itertuples():
        if on_step:
            on_step(f"Downloading wave forecast: {s.name}")
        params = {"latitude": s.forecast_lat, "longitude": s.forecast_lon, "hourly": ",".join(MARINE_VARS),
                  "timezone": TIMEZONE, "forecast_days": FORECAST_DAYS}
        data, err = get_json(session, MARINE_URL, params, f"marine[{s.spot_id}]")
        if err:
            failed[s.spot_id] = err
            continue
        df = _hourly_frame(data, s.spot_id, MARINE_VARS)
        if df["wave_height"].isna().all():
            msg = f"marine[{s.spot_id}]: all wave values empty; the point is probably a land cell. Set forecast_lat/lon further offshore."
            log.warning(msg)
            failed[s.spot_id] = msg
        frames.append(df)
    out = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=["spot_id", "time", *MARINE_VARS])
    return out, failed


def fetch_wind(session, spots, on_step=None):
    frames, failed = [], {}
    for s in spots.itertuples():
        if on_step:
            on_step(f"Downloading wind forecast: {s.name}")
        params = {"latitude": s.lat, "longitude": s.lon, "hourly": ",".join(WIND_VARS),
                  "wind_speed_unit": "kmh", "timezone": TIMEZONE, "forecast_days": FORECAST_DAYS}
        data, err = get_json(session, WEATHER_URL, params, f"wind[{s.spot_id}]")
        if err:
            failed[s.spot_id] = err
            continue
        frames.append(_hourly_frame(data, s.spot_id, WIND_VARS))
    out = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=["spot_id", "time", *WIND_VARS])
    return out, failed

## 5. CWA buoy and tide-station observations
Observations are our ground truth: they show *current* conditions in the demo and let us measure forecast error in `05_evaluation`.

**Why a generic parser?** We cannot assume the exact JSON layout of every CWA dataset version. The parser walks the JSON tree, turns every time-stamped record into one row (carrying the station fields of its parent objects), and then maps field names to our English column names with the patterns in `CWA_FIELD_PATTERNS`.

**Checked against a real response (2026-10-16):** the top-level key is `Records`, stations carry only a `StationID`, and the field names match the patterns below. Units: wave height m, wave period s, wave direction degrees, tide height m (values about −3 to +3), wind speed m/s. If CWA changes its format, a saved `raw/cwa_obs_48h.json.gz` shows the new layout; the raw files are kept so earlier snapshots can be re-parsed.

In [ ]:
import re

# English column name -> regex matched against the last part of the flattened CWA field name (first match wins)
CWA_FIELD_PATTERNS = {
    "station_id": r"^StationID$",
    "station_name_source": r"^StationName$",
    "obs_time": r"^(DateTime|ObsTime|DataTime)$",
    "station_lat": r"^(StationLatitude|Latitude)$",
    "station_lon": r"^(StationLongitude|Longitude)$",
    "wave_height_m": r"^WaveHeight$",
    "wave_dir_deg": r"^WaveDirection$",
    "wave_period_s": r"^WavePeriod$",
    "tide_height_m": r"^TideHeight$",
    "wind_speed_ms": r"^WindSpeed$",
    "wind_dir_deg": r"^WindDirection$",
    "sea_temp_c": r"^SeaTemperature$",
}
CWA_UNIT_FACTORS = {"tide_height_m": 1.0}   # tide height is already in metres (checked 2026-10-16)
CWA_NUMERIC = ["station_lat", "station_lon", "wave_height_m", "wave_dir_deg", "wave_period_s",
               "tide_height_m", "wind_speed_ms", "wind_dir_deg", "sea_temp_c"]
TIME_KEYS = {"DateTime", "ObsTime", "DataTime"}


def _scalars(d, prefix=""):
    """Flatten nested dicts into {'A.B.C': value}, skipping lists."""
    out = {}
    for k, v in d.items():
        if isinstance(v, dict):
            out.update(_scalars(v, f"{prefix}{k}."))
        elif not isinstance(v, list):
            out[f"{prefix}{k}"] = v
    return out


def _walk(node, context, rows):
    if isinstance(node, dict):
        if TIME_KEYS & set(node.keys()):          # a time-stamped observation record
            rows.append({**context, **_scalars(node)})
            return
        ctx = {**context, **_scalars(node)}
        for v in node.values():
            if isinstance(v, (dict, list)):
                _walk(v, ctx, rows)
    elif isinstance(node, list):
        for item in node:
            _walk(item, context, rows)


def parse_cwa_observations(raw):
    rows = []
    _walk(raw.get("Records", raw.get("records", raw)), {}, rows)
    if not rows:
        return pd.DataFrame(columns=list(CWA_FIELD_PATTERNS))
    flat = pd.DataFrame(rows)
    out = pd.DataFrame(index=flat.index)
    for target, pattern in CWA_FIELD_PATTERNS.items():
        matches = [c for c in flat.columns if re.search(pattern, c.split(".")[-1])]
        out[target] = flat[matches].bfill(axis=1).iloc[:, 0] if matches else np.nan
    for col in CWA_NUMERIC:
        out[col] = pd.to_numeric(out[col], errors="coerce") * CWA_UNIT_FACTORS.get(col, 1.0)
    times = pd.to_datetime(out["obs_time"], errors="coerce", utc=True)
    out["obs_time"] = times.dt.tz_convert(TIMEZONE).dt.tz_localize(None)
    out["station_id"] = out["station_id"].astype(str)
    return out.dropna(subset=["obs_time"]).drop_duplicates(["station_id", "obs_time"]).reset_index(drop=True)


def fetch_cwa(session, dataset_key):
    dataset_id = CWA_DATASETS[dataset_key]
    params = {"Authorization": CWA_API_KEY, "format": "JSON"}
    return get_json(session, CWA_URL.format(dataset_id=dataset_id), params, f"cwa[{dataset_key}={dataset_id}]")

### English station names
The observation data contains only station IDs. CWA also publishes an official station list (`O-B0076-001`) with **English names**, station type and coordinates. On each run we refresh `cwa_stations_en.csv` from that list, keeping any notes the team added. The English names are CWA's own, so nothing is translated by us. Stations that appear in the observations but not in the list go to `cwa_stations_unmapped.csv`.

In [ ]:
def station_type(attribute_en):
    a = str(attribute_en).lower()
    return "buoy" if "buoy" in a else ("tide" if "tid" in a else "met")


def refresh_station_lookup(session):
    """Download CWA's station list and rewrite cwa_stations_en.csv (team notes are preserved). Returns the raw JSON."""
    raw, err = get_json(session, CWA_STATIONS_URL, {"Authorization": CWA_API_KEY, "downloadType": "WEB", "format": "JSON"},
                        "cwa[stations=O-B0076-001]")
    if err:
        return None
    locations = raw["cwaopendata"]["Resources"]["Resource"]["Data"]["SeaSurfaceObs"]["Location"]
    rows = [{"station_id": str(l["Station"]["StationID"]), "station_name_source": l["Station"].get("StationName"),
             "station_name_en": l["Station"].get("StationNameEN"), "station_type": station_type(l["Station"].get("StationAttributeEN")),
             "lat": pd.to_numeric(l["Station"].get("StationLatitude"), errors="coerce"),
             "lon": pd.to_numeric(l["Station"].get("StationLongitude"), errors="coerce")} for l in locations]
    lookup = pd.DataFrame(rows)
    old = load_station_lookup()
    notes = old.set_index("station_id")["notes"] if "notes" in old else pd.Series(dtype=str)
    lookup["notes"] = lookup["station_id"].map(notes).fillna("")
    lookup.sort_values("station_id").to_csv(STATION_LOOKUP_FILE, index=False, encoding="utf-8")
    log.info(f"Station lookup refreshed: {len(lookup)} stations ({(lookup['station_type'] == 'buoy').sum()} buoys)")
    return raw


def load_station_lookup():
    if not STATION_LOOKUP_FILE.exists():
        return pd.DataFrame(columns=["station_id", "station_name_en", "station_type", "lat", "lon"])
    return pd.read_csv(STATION_LOOKUP_FILE, dtype={"station_id": str})


def apply_station_names(obs):
    lookup = load_station_lookup()[["station_id", "station_name_en", "station_type"]]
    obs = obs.drop(columns=["station_name_en", "station_type"], errors="ignore").merge(lookup, on="station_id", how="left")
    unmapped = obs.loc[obs["station_name_en"].isna(), ["station_id", "station_name_source", "station_lat", "station_lon"]]
    unmapped = unmapped.drop_duplicates("station_id")
    if len(unmapped):
        known = pd.read_csv(UNMAPPED_STATIONS_FILE, dtype={"station_id": str}) if UNMAPPED_STATIONS_FILE.exists() else unmapped.iloc[0:0]
        merged = pd.concat([known, unmapped]).drop_duplicates("station_id").sort_values("station_id")
        merged.to_csv(UNMAPPED_STATIONS_FILE, index=False, encoding="utf-8")
        log.info(f"{len(unmapped)} CWA stations have no English name yet; listed in {UNMAPPED_STATIONS_FILE.name}")
    return obs

## 6. One function that collects everything
`collect_all()` returns the data in memory. The notebook (and the scheduled script) then saves it; `06_demo` calls the same function for its *live data* mode, without saving. The optional `progress(done, total, label)` callback lets the demo show a progress bar while the ~70 requests run.

In [ ]:
def collect_all(spots=None, progress=None):
    spots = load_spot_locations() if spots is None else spots
    session = make_session()
    started = time.perf_counter()
    use_cwa = bool(FETCH_CWA and CWA_API_KEY)
    # Progress weights: one unit per forecast request (~0.4 s each); the CWA downloads are large and often slow
    # (measured 2026-10-16: station list ~20 s, 48-h observations ~40 s), so they count as many units.
    W_STATIONS, W_OBS = 30, 60
    total = 2 * len(spots) + ((W_STATIONS if REFRESH_STATION_LOOKUP else 0) + W_OBS * len(CWA_DATASETS) if use_cwa else 0)
    counter = {"done": 0, "total": total}

    def step(label, weight=1):
        if progress:
            progress(counter["done"], counter["total"], label)
        counter["done"] += weight

    marine, marine_failed = fetch_marine(session, spots, on_step=step)
    wind, wind_failed = fetch_wind(session, spots, on_step=step)
    obs, cwa_raw, cwa_status = pd.DataFrame(), {}, "skipped (no API key or FETCH_CWA=False)"
    if FETCH_CWA and CWA_API_KEY:
        if REFRESH_STATION_LOOKUP:
            step("Downloading CWA station list (can take 20 s or more)", W_STATIONS)
            stations_raw = refresh_station_lookup(session)
            if stations_raw is not None:
                cwa_raw["stations"] = stations_raw
        cwa_status, frames = "ok", []
        for key in CWA_DATASETS:
            step("Downloading CWA buoy and tide observations (large file, can take up to a minute)", W_OBS)
            raw, err = fetch_cwa(session, key)
            if err:
                cwa_status = f"failed: {err}"
                continue
            cwa_raw[key] = raw
            frames.append(parse_cwa_observations(raw).assign(dataset=key))
        if frames:
            obs = apply_station_names(pd.concat(frames, ignore_index=True))
            if obs.empty:
                cwa_status = "parsed 0 rows: check CWA_FIELD_PATTERNS against the raw JSON"
    meta = {
        "fetched_at": now_taiwan().isoformat(),
        "timezone": TIMEZONE,
        "n_spots": int(len(spots)),
        "marine_failed": marine_failed,
        "wind_failed": wind_failed,
        "cwa_status": cwa_status,
        "cwa_datasets": CWA_DATASETS,
        "n_obs_rows": int(len(obs)),
        "duration_s": round(time.perf_counter() - started, 1),
        "attribution": [
            "Marine and weather forecasts: Open-Meteo.com (CC BY 4.0), incl. DWD wave model data",
            "Observations: Central Weather Administration (CWA) Open Data, Taiwan",
        ],
        "parameters": {"forecast_days": FORECAST_DAYS, "marine_vars": MARINE_VARS, "wind_vars": WIND_VARS},
    }
    ok_marine = len(spots) - len(marine_failed)
    meta["status"] = "failed" if ok_marine == 0 else ("complete" if not (marine_failed or wind_failed) and cwa_status == "ok" else "partial")
    return {"marine": marine, "wind": wind, "obs": obs, "cwa_raw": cwa_raw, "meta": meta}

## 7. Run the collection
The folder name is the Taiwan *local* date. If today's snapshot already exists, we skip (set `OVERWRITE = True` to replace it). Data is first written to a temporary folder and then renamed, so a crash never leaves a half-written snapshot that looks complete.

In [ ]:
snapshot_date = SNAPSHOT_DATE or now_taiwan().date().isoformat()
snapshot_dir = SNAP_ROOT / snapshot_date
RUN_COLLECTION = not LIBRARY_MODE and (OVERWRITE or not (snapshot_dir / "manifest.json").exists())
SNAPSHOT_STATUS = "skipped"
if LIBRARY_MODE:
    print("Library mode: functions defined, nothing fetched.")
elif not RUN_COLLECTION:
    print(f"Snapshot {snapshot_date} already exists; skipping (set OVERWRITE = True to refetch).")
else:
    print(f"Collecting snapshot {snapshot_date} -> {snapshot_dir.relative_to(ROOT)}")

In [ ]:
if RUN_COLLECTION:
    tmp_dir = SNAP_ROOT / f"{snapshot_date}.tmp"
    shutil.rmtree(tmp_dir, ignore_errors=True)
    attach_log_file(tmp_dir / "collection.log")
    snap = collect_all()
    log.info(f"status={snap['meta']['status']} marine_rows={len(snap['marine'])} wind_rows={len(snap['wind'])} obs_rows={len(snap['obs'])}")

In [ ]:
if RUN_COLLECTION:
    (tmp_dir / "raw").mkdir(parents=True, exist_ok=True)
    snap["marine"].to_parquet(tmp_dir / "marine.parquet", index=False)
    snap["wind"].to_parquet(tmp_dir / "wind.parquet", index=False)
    if not snap["obs"].empty:
        snap["obs"].to_parquet(tmp_dir / "cwa_obs.parquet", index=False)
    for key, raw in snap["cwa_raw"].items():
        with gzip.open(tmp_dir / "raw" / f"cwa_{key}.json.gz", "wt", encoding="utf-8") as f:
            json.dump(raw, f, ensure_ascii=False)
    (tmp_dir / "manifest.json").write_text(json.dumps(snap["meta"], indent=2, ensure_ascii=False), encoding="utf-8")
    attach_log_file(SNAP_ROOT / "_last_run.log")      # release the file handle before renaming
    if snapshot_dir.exists():
        shutil.rmtree(snapshot_dir)
    tmp_dir.rename(snapshot_dir)
    SNAPSHOT_STATUS = snap["meta"]["status"]
    print(f"Saved snapshot {snapshot_date} (status: {SNAPSHOT_STATUS})")

## 8. Quick check
A sanity check you should glance at daily during the first week: Are all spots present? Are wave heights plausible (no empty spots)? Did CWA work?

In [ ]:
if not LIBRARY_MODE and (snapshot_dir / "manifest.json").exists():
    manifest = json.loads((snapshot_dir / "manifest.json").read_text(encoding="utf-8"))
    print({k: manifest[k] for k in ["fetched_at", "status", "n_spots", "cwa_status", "n_obs_rows", "duration_s"]})
    marine_check = pd.read_parquet(snapshot_dir / "marine.parquet")
    display(marine_check.groupby("spot_id")[["wave_height", "swell_wave_period", "sea_level_height_msl"]]
            .agg(["min", "max"]).round(2))
    print("Snapshots on disk:", sorted(p.name for p in SNAP_ROOT.iterdir() if p.is_dir() and not p.name.endswith(".tmp")))